# 01. Кластеризация: распределение точек по базам

Первый шаг схемы «cluster-first, route-second». Задача: раздать точки
обслуживания по базам так, чтобы не превысить мощность ни одной, а затем
раздробить крупные группы до размера, посильного солверу.

Данные синтетические — генерируются `scripts/make_sample_data.py`.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from routeforge.clustering import UNASSIGNED, assign_to_depots, optimal_k_by_cluster_size
from routeforge.distance import haversine_matrix
from routeforge.io import read_points

sites = read_points("../data/sample/sites.csv")
depots_df = pd.read_csv("../data/sample/depots.csv")
depots = list(depots_df[["lat", "lon"]].itertuples(index=False, name=None))

print(f"{len(sites)} точек, {len(depots)} баз, суммарный спрос {int(sites.demand.sum()):,}")
sites.head()

## Расстояния до баз

Для распределения по базам достаточно haversine: важен порядок близости,
а он совпадает с дорожным. Дорожные расстояния нужны позже, при построении
самих маршрутов.

In [ ]:
site_coords = list(sites[["lat", "lon"]].itertuples(index=False, name=None))
to_depots = haversine_matrix(site_coords, depots)

print("матрица:", to_depots.shape)
print("до ближайшей базы, км: медиана %.1f, максимум %.1f" % (
    np.median(to_depots.min(axis=1)) / 1000, to_depots.min(axis=1).max() / 1000))

## Раздача без ограничения мощности

Каждая точка уходит на ближайшую базу — простейший случай.

In [ ]:
free = assign_to_depots(to_depots)

for d in range(len(depots)):
    mask = free == d
    print(f"База {d} ({depots_df.name[d]}): {mask.sum():3} точек, "
          f"спрос {int(sites.demand[mask].sum()):,}")

## Раздача с ограничением мощности

Теперь у баз есть предел. Точки обходятся в порядке возрастания расстояния
до ближайшей базы: у кого выбор беднее, тот занимает место первым. Если ни
одна база не может принять точку, она помечается `UNASSIGNED` и в маршруты
не попадает — молча терять её нельзя.

Зажмём мощности так, чтобы их заведомо не хватило, и посмотрим на поведение.

In [ ]:
tight = [8000, 8000, 8000]   # суммарно 24 000 при спросе ~27 500

limited = assign_to_depots(to_depots, demands=sites.demand.to_numpy(), capacities=tight)

for d in range(len(depots)):
    mask = limited == d
    print(f"База {d}: {mask.sum():3} точек, спрос {int(sites.demand[mask].sum()):,} / {tight[d]:,}")
print(f"\nне распределено: {(limited == UNASSIGNED).sum()} точек")

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 5))

for ax, labels, title in ((a, free, "без ограничений"), (b, limited, "с ограничением мощности")):
    for d in range(len(depots)):
        m = labels == d
        ax.scatter(sites.lon[m], sites.lat[m], s=10, alpha=0.75, label=f"База {d}")
    m = labels == UNASSIGNED
    if m.any():
        ax.scatter(sites.lon[m], sites.lat[m], s=14, c="0.6", marker="x", label="не распределены")
    ax.scatter([d[1] for d in depots], [d[0] for d in depots], s=180, c="k", marker="*", zorder=5)
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
plt.tight_layout()

Видно главное свойство: при нехватке мощности отсекаются не случайные
точки, а самые дальние — те, что обходятся дороже всего.

## Дробление крупных групп

Время решения CVRP растёт быстрее линейного, поэтому группу в тысячи точек
надо разбить. `optimal_k_by_cluster_size` ищет наименьшее `k`, при котором
в каждом кластере не больше заданного числа точек.

In [ ]:
block = sites[free == 0]
coords = np.asarray(list(block[["lat", "lon"]].itertuples(index=False, name=None)))

for limit in (20, 40, 60, 100):
    k, labels = optimal_k_by_cluster_size(coords, max_per_cluster=limit)
    sizes = np.bincount(labels)
    print(f"не более {limit:3} точек -> k = {k}, размеры {sizes.tolist()}")

In [ ]:
k, labels = optimal_k_by_cluster_size(coords, max_per_cluster=40)

plt.figure(figsize=(7, 5.5))
for c in range(k):
    m = labels == c
    plt.scatter(coords[m, 1], coords[m, 0], s=14, label=f"кластер {c}")
plt.scatter(depots[0][1], depots[0][0], s=200, c="k", marker="*", zorder=5)
plt.title(f"База 0 разбита на {k} групп не более чем по 40 точек")
plt.legend(fontsize=8)
plt.grid(alpha=0.2)

Дальше каждая такая группа уходит в солвер отдельно — см.
[03_cvrp.ipynb](03_cvrp.ipynb).